# Tải và kiểm tra dữ liệu PhoNER_COVID19

Notebook này tải bộ dữ liệu **PhoNER_COVID19** từ repository chính thức của VinAI Research:  
https://github.com/VinAIResearch/PhoNER_COVID19

Chỉ trích xuất thư mục `data/` vào workspace:
- `data/word/`: Dữ liệu tách từ (word-level) gồm `train_word`, `dev_word`, `test_word` (định dạng `.json` và `.conll`)
- `data/syllable/`: Dữ liệu tách âm tiết (syllable-level) gồm `train_syllable`, `dev_syllable`, `test_syllable` (định dạng `.json` và `.conll`)

In [2]:
import io
import json
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

REPO_ZIP_URL = "https://github.com/VinAIResearch/PhoNER_COVID19/archive/refs/heads/main.zip"
DATA_DIR = Path("data")
FORCE_DOWNLOAD = False

In [3]:
def download_phoner_covid19_data(zip_url: str, output_dir: Path, force: bool = False) -> Path:
    """Tải và chỉ trích xuất thư mục data của PhoNER_COVID19 vào output_dir."""
    if output_dir.exists() and any(output_dir.iterdir()) and not force:
        print(f"Thư mục '{output_dir}' đã tồn tại. Bỏ qua bước tải (đặt FORCE_DOWNLOAD = True để tải lại).")
        return output_dir

    print(f"Đang tải dữ liệu từ: {zip_url}")
    with urllib.request.urlopen(zip_url, timeout=60) as resp:
        raw_bytes = resp.read()
    print(f"Đã tải {len(raw_bytes) / (1024 * 1024):.2f} MB. Đang giải nén chỉ riêng thư mục 'data'...")

    output_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as zf:
        prefix = "PhoNER_COVID19-main/data/"
        for member in zf.infolist():
            if member.is_dir() or not member.filename.startswith(prefix):
                continue
            # Lấy đường dẫn tương đối bên trong thư mục 'data/'
            rel_path = member.filename[len(prefix):]
            target_path = output_dir / rel_path
            target_path.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, open(target_path, "wb") as dst:
                dst.write(src.read())

    print(f"Hoàn tất giải nén vào: {output_dir.resolve()}")
    return output_dir

download_phoner_covid19_data(REPO_ZIP_URL, DATA_DIR, force=FORCE_DOWNLOAD)

Thư mục 'data' đã tồn tại. Bỏ qua bước tải (đặt FORCE_DOWNLOAD = True để tải lại).


WindowsPath('data')

In [4]:
print("Danh sách các tệp trong thư mục data:")
data_files = sorted(DATA_DIR.rglob("*"))
for file_path in data_files:
    if file_path.is_file():
        size_kb = file_path.stat().st_size / 1024
        print(f"  - {file_path} ({size_kb:.1f} KB)")

Danh sách các tệp trong thư mục data:
  - data\syllable\dev_syllable.conll (699.9 KB)
  - data\syllable\dev_syllable.json (1159.6 KB)
  - data\syllable\test_syllable.conll (1066.4 KB)
  - data\syllable\test_syllable.json (1763.9 KB)
  - data\syllable\train_syllable.conll (1560.4 KB)
  - data\syllable\train_syllable.json (2646.9 KB)
  - data\word\dev_word.conll (613.5 KB)
  - data\word\dev_word.json (985.0 KB)
  - data\word\test_word.conll (935.6 KB)
  - data\word\test_word.json (1500.2 KB)
  - data\word\train_word.conll (1385.5 KB)
  - data\word\train_word.json (2266.8 KB)


In [6]:
def load_jsonl(path: Path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

for level in ["word", "syllable"]:
    print(f"\nMức {level.upper()}")
    for split in ["train", "dev", "test"]:
        file_path = DATA_DIR / level / f"{split}_{level}.json"
        samples = load_jsonl(file_path)
        tag_counts = Counter(tag for sample in samples for tag in sample["tags"])
        entity_types = sorted({tag.split("-", 1)[1] for tag in tag_counts if "-" in tag})
        total_tokens = sum(len(sample["words"]) for sample in samples)
        print(f"{split:5s}: {len(samples):5d} câu | {total_tokens:6d} tokens | {len(entity_types)} loại thực thể")

# Quét toàn bộ tập train để lấy tất cả các nhãn và in ra tên nhãn
sample_train = load_jsonl(DATA_DIR / "word" / "train_word.json")
labels = sorted({tag.split("-", 1)[1] for sample in sample_train for tag in sample["tags"] if "-" in tag})

print(f"\nDanh sách các tên nhãn ({len(labels)} nhãn):\n")
for label in labels:
    print(label)


Mức WORD
train:  5027 câu | 132511 tokens | 10 loại thực thể
dev  :  2000 câu |  56283 tokens | 10 loại thực thể
test :  3000 câu |  85678 tokens | 10 loại thực thể

Mức SYLLABLE
train:  5027 câu | 167541 tokens | 10 loại thực thể
dev  :  2000 câu |  71325 tokens | 10 loại thực thể
test :  3000 câu | 108354 tokens | 10 loại thực thể

Danh sách các tên nhãn (10 nhãn):

AGE
DATE
GENDER
JOB
LOCATION
NAME
ORGANIZATION
PATIENT_ID
SYMPTOM_AND_DISEASE
TRANSPORTATION
